# Agent selection, veto, and series forecasts

Audit legal agent probabilities, pre/post-veto series scores, and whether the pooled sequential veto model beats a uniform choice baseline.

This notebook is generated from immutable, hash-manifested artifacts. Any
counterfactual agent or composition result is predictive/associational,
not a causal estimate.

In [1]:
from pathlib import Path

SNAPSHOT = Path(r"C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\data\snapshots\vct_2023_2026_cutoff_2026-06-21_v2")
DEVELOPMENT = Path(r"C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\artifacts\ratings-development-v1-full")
CONFIRMATION = Path(r"C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\artifacts\ratings-development-v1-full__2026-confirmation")
SCORING_V3 = Path(r"C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\artifacts\ratings-development-v1-full__rating-snapshots-v3")
SCORING_V2 = Path(r"C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\artifacts\ratings-development-v1-full__rating-snapshots-v2")
SCORING_V1 = Path(r"C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\artifacts\ratings-development-v1-full__rating-snapshots")
SCORING = SCORING_V3 if SCORING_V3.exists() else (SCORING_V2 if SCORING_V2.exists() else SCORING_V1)

print("snapshot:", SNAPSHOT)
print("development:", DEVELOPMENT if DEVELOPMENT.exists() else "not supplied")
print("confirmation:", CONFIRMATION if CONFIRMATION.exists() else "not available")
print("scoring:", SCORING if SCORING.exists() else "not available")

snapshot: C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\data\snapshots\vct_2023_2026_cutoff_2026-06-21_v2
development: C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\artifacts\ratings-development-v1-full
confirmation: C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\artifacts\ratings-development-v1-full__2026-confirmation
scoring: C:\Users\benja\projects\valorant-elo-dashboard\research\ratings\artifacts\ratings-development-v1-full__rating-snapshots-v3


In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

probabilities = pd.read_parquet(SCORING / "agent_selection_probabilities.parquet")
veto = pd.read_parquet(DEVELOPMENT / "metrics" / "veto.parquet")
series = pd.read_parquet(DEVELOPMENT / "metrics" / "series.parquet")

sums = probabilities.groupby(["player_id", "team_id", "map_name"])["probability"].sum()
print("maximum probability-sum error:", float((sums - 1).abs().max()))
assert np.allclose(sums, 1.0)
display(series)
display(veto)

maximum probability-sum error: 4.440892098500626e-16


,model,series,series_log_loss,series_brier
0,carryover_pre_veto,930,0.654180,0.231449
1,carryover_post_veto,930,0.654205,0.231459


,model,matches,choice_log_loss,uniform_choice_log_loss
0,sequential_pooled_veto,930,1.556083,1.420511


In [3]:
probabilities["entropy_term"] = -probabilities["probability"] * np.log(
    probabilities["probability"].clip(lower=1e-12)
)
entropy = probabilities.groupby(["player_id", "team_id", "map_name"])["entropy_term"].sum()
display(entropy.describe().rename("agent_assignment_entropy"))

example_key = probabilities.groupby(["player_id", "team_id", "map_name"]).size().index[0]
example = probabilities.set_index(["player_id", "team_id", "map_name"]).loc[example_key]
display(example.nlargest(10, "probability")[["agent", "probability"]])

row = veto.iloc[0]
if row["choice_log_loss"] >= row["uniform_choice_log_loss"]:
    print("Decision: reject the current veto model; it underperforms uniform choice log loss.")
else:
    print("Decision: veto model improves on uniform choice log loss.")

count    1680.000000
mean        2.730534
std         0.262323
min         1.501573
25%         2.584118
50%         2.768237
75%         2.926246
max         3.227521
Name: agent_assignment_entropy, dtype: float64

C:\Users\benja\AppData\Local\Temp\ipykernel_6936\2553820282.py:8: PerformanceWarning: indexing past lexsort depth may impact performance.
  example = probabilities.set_index(["player_id", "team_id", "map_name"]).loc[example_key]


agent  probability
player_id team_id map_name                     
4         27      Ascent      Fade     0.180422
                  Ascent      Omen     0.166386
                  Ascent      Sova     0.110947
                  Ascent     KAY/O     0.094684
                  Ascent      Skye     0.085565
                  Ascent    Breach     0.030693
                  Ascent      Raze     0.027339
                  Ascent      Yoru     0.024918
                  Ascent    Harbor     0.024497
                  Ascent      Tejo     0.021047

Decision: reject the current veto model; it underperforms uniform choice log loss.


Series simulations use shared latent strength draws so map outcomes are
correlated. A veto model is useful only when its held-out action log loss
improves on the legal uniform baseline; otherwise pre-veto integration
should retain a simpler map prior.